# EDA de verificación — train_2016_2017.csv

El objetivo de este notebook **no** es un EDA completo, sino confirmar que el filtrado se hizo bien: que solo quedaron los años 2016-2017, que están todas las tiendas, que no se duplicaron filas, y una mirada rápida a nulos y a `unit_sales`.

In [ ]:
import pandas as pd
import os

BASE_DIR = "C:/Tesis"


## 1. Cargar train_2016_2017.csv
Le especificamos los tipos de dato de antemano (como en el notebook que lo generó), para que cargue más rápido y use menos memoria.

In [ ]:
FILE_PATH = os.path.join(BASE_DIR, "train_2016_2017.csv")

tamano_gb = os.path.getsize(FILE_PATH) / (1024**3)
print(f"Tamaño en disco: {tamano_gb:.2f} GB")


In [ ]:
DTYPES = {
    "id": "int64",
    "store_nbr": "int16",
    "item_nbr": "int32",
    "unit_sales": "float32",
}

df = pd.read_csv(FILE_PATH, dtype=DTYPES, parse_dates=["date"])
print(f"Filas: {len(df):,}")
print(f"Columnas: {df.shape[1]}")


## 2. Forma general

In [ ]:
df.info()


In [ ]:
df.head(10)


## 3. ¿Quedaron solo 2016 y 2017?
Esta es la verificación más importante: `años_presentes` debería mostrar exactamente `[2016, 2017]`, nada más.

In [ ]:
anios_presentes = sorted(df["date"].dt.year.unique())
print("Años presentes en el archivo:", anios_presentes)
print("Fecha mínima:", df["date"].min())
print("Fecha máxima:", df["date"].max())

assert anios_presentes == [2016, 2017], "⚠️ Hay años que no deberían estar — revisa el filtro"
print("\n✅ Solo están 2016 y 2017.")


## 4. ¿Están todas las tiendas?
Comparamos las tiendas que aparecen en `train_2016_2017` contra la lista completa de `stores.csv`.

In [ ]:
stores = pd.read_csv(os.path.join(BASE_DIR, "stores.csv"))

tiendas_en_train = set(df["store_nbr"].unique())
tiendas_en_stores = set(stores["store_nbr"].unique())

print("Tiendas distintas en train_2016_2017:", len(tiendas_en_train))
print("Tiendas distintas en stores.csv:", len(tiendas_en_stores))

faltantes = tiendas_en_stores - tiendas_en_train
print("Tiendas que están en stores.csv pero NO aparecen en train_2016_2017:", sorted(faltantes) if faltantes else "ninguna")


## 5. Nulos por columna
`onpromotion` normalmente tiene nulos (no todas las filas indican si el producto estaba en promoción). El resto de las columnas debería tener 0 o muy pocos nulos.

In [ ]:
df.isna().sum()


## 6. ¿Hay filas duplicadas?
Revisamos duplicados por `id` (debería ser único, es la clave del archivo original) y por la combinación tienda + producto + fecha (no debería repetirse).

In [ ]:
dup_id = df["id"].duplicated().sum()
dup_combo = df.duplicated(subset=["store_nbr", "item_nbr", "date"]).sum()

print("Filas con id duplicado:", dup_id)
print("Filas con combinación (tienda, producto, fecha) duplicada:", dup_combo)


## 7. unit_sales — estadísticas generales
Ojo: es normal encontrar valores **negativos** en `unit_sales` — representan devoluciones, no son un error de datos.

In [ ]:
df["unit_sales"].describe()


In [ ]:
negativos = (df["unit_sales"] < 0).sum()
print(f"Filas con unit_sales negativo (devoluciones): {negativos:,} ({negativos/len(df)*100:.2f}%)")


## 8. onpromotion — valores

In [ ]:
df["onpromotion"].value_counts(dropna=False)


## 9. Resumen final
Un check rápido de todo lo anterior, para revisar de un vistazo si el filtrado quedó bien.

In [ ]:
print("===== RESUMEN DE VERIFICACIÓN =====")
print(f"Filas totales: {len(df):,}")
print(f"Rango de fechas: {df['date'].min().date()} -> {df['date'].max().date()}")
print(f"Años presentes: {sorted(df['date'].dt.year.unique())}")
print(f"Tiendas presentes: {df['store_nbr'].nunique()} de {stores['store_nbr'].nunique()} en stores.csv")
print(f"Productos distintos: {df['item_nbr'].nunique():,}")
print(f"IDs duplicados: {dup_id}")
print(f"Combinaciones (tienda, producto, fecha) duplicadas: {dup_combo}")
print(f"Nulos en unit_sales: {df['unit_sales'].isna().sum()}")
print(f"Nulos en onpromotion: {df['onpromotion'].isna().sum():,} ({df['onpromotion'].isna().mean()*100:.1f}%)")
